# HaaS vs local-reduce histogram comparison

Run the Z' → tt̄ single-lepton analysis twice over identical inputs — once with `HistServProcessor` (histograms filled on a remote histserv via gRPC) and once with `HistLocalProcessor` (histograms filled on workers and reduced by coffea). Report side-by-side metrics and verify the final histograms match bin-by-bin.

## AF flag

In [ ]:
AF = "coffeacasa-gateway"  # options: [coffeacasa-condor, coffeacasa-gateway, purdue-af-k8s, purdue-af-slurm]
AUTO_CLOSE_CLIENT = False

## Imports and dependencies

### The intccms package
Add `src/` and the repo root to `sys.path` so the `intccms` and `example_cms` packages are importable without installation.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
src_dir = repo_root / "src"
examples_dir = repo_root
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if str(examples_dir) not in sys.path:
    sys.path.insert(0, str(examples_dir))
print(f"✅ Added {src_dir} to Python path")
print(f"✅ Added {examples_dir} to Python path")

### Installing extra dependencies
Install `omegaconf`, `roastcoffea`, and `histserv` on the client. `histserv` is needed so the HaaS arm can talk to the server.

In [ ]:
import subprocess, sys, importlib

def pip_install(spec):
    try:
        subprocess.check_output(
            [sys.executable, "-m", "pip", "install", "-q", spec],
            stderr=subprocess.STDOUT,
        )
        print(f"✅ {spec}")
    except subprocess.CalledProcessError as e:
        print(f"❌ {spec}\n{e.output.decode()}")

def ensure(pkg, spec=None, version=None):
    try:
        mod = importlib.import_module(pkg)
        if version and getattr(mod, "__version__", None) != version:
            raise ImportError
        print(f"✓ {pkg} already installed")
    except ImportError:
        pip_install(spec or pkg)

ensure("omegaconf")
ensure("roastcoffea", "roastcoffea==0.1.2", "0.1.2")
ensure("histserv", "histserv==0.1.9", "0.1.9")

### Alternative coffea version
Pin the coffea version on both the client and the Dask workers.

In [ ]:
COFFEA_VERSION = "2025.12.0"
COFFEA_PIP = COFFEA_VERSION if "git" in COFFEA_VERSION else f"coffea=={COFFEA_VERSION}"
WORKER_DEPENDENCIES = [COFFEA_PIP, "roastcoffea==0.1.2", "histserv==0.1.9"]

pip_install(COFFEA_PIP)

### Imports from stdlib and other libraries

In [ ]:
import cloudpickle
import copy
import time

from coffea.processor import DaskExecutor
from coffea.nanoevents import NanoAODSchema

### Imports from intccms

In [ ]:
from intccms.schema import Config, load_config_with_restricted_cli
from intccms.utils.output import OutputDirectoryManager
from intccms.metadata_extractor import DatasetMetadataManager
from intccms.datasets import DatasetManager
from intccms.analysis import (
    run_processor_workflow,
    HistServProcessor,
    HistLocalProcessor,
)

from roastcoffea import MetricsCollector

### Registering packages with cloudpickle

In [ ]:
import intccms
import example_cms

cloudpickle.register_pickle_by_value(intccms)
cloudpickle.register_pickle_by_value(example_cms)

## Dask client setup

In [ ]:
from intccms.utils.dask_client import acquire_client

## Configuration
Same config for both arms — we want the only difference to be the histogram backend.

In [ ]:
from example_cms.configs.configuration import config as original_config

config = copy.deepcopy(original_config)

config["datasets"]["max_files"] = 5  # small, for the comparison loop
config["general"]["output_dir"] = "example_cms/outputs/"
config["general"]["run_metadata_generation"] = False  # reuse cache
config["general"]["run_processor"] = True
config["general"]["run_analysis"] = True
config["general"]["save_skimmed_output"] = False
config["general"]["run_histogramming"] = True
config["general"]["run_systematics"] = False
config["general"]["run_statistics"] = False

full_config = load_config_with_restricted_cli(config, [])
validated_config = Config(**full_config)

## Output manager

In [ ]:
output_manager = OutputDirectoryManager(
    root_output_dir=validated_config.general.output_dir,
    cache_dir=validated_config.general.cache_dir,
    metadata_dir=validated_config.general.metadata_dir,
    skimmed_dir=validated_config.general.skimmed_dir,
)

## Redirector override

In [ ]:
REDIRECTOR = "root://xcache/"
for dataset in validated_config.datasets.datasets:
    dataset.redirector = REDIRECTOR
print(f"Redirector: {REDIRECTOR}")

## Dataset manager and metadata

In [ ]:
dataset_manager = DatasetManager(validated_config.datasets)

metadata_generator = DatasetMetadataManager(
    dataset_manager=dataset_manager,
    output_manager=output_manager,
    config=validated_config,
)

if metadata_generator.generate_metadata:
    with acquire_client(
        AF, close_after=AUTO_CLOSE_CLIENT, pip_packages=WORKER_DEPENDENCIES
    ) as (client, cluster):
        metadata_generator.run(executor=DaskExecutor(client=client))
else:
    metadata_generator.run()

metadata_lookup = metadata_generator.build_metadata_lookup()
workitems = metadata_generator.workitems
print(f"{len(workitems)} workitems loaded")

## Run both arms
One client context, two processor runs back-to-back. The helper below wraps each run in a `MetricsCollector` so we can compare.

In [ ]:
def run_arm(client, processor, label):
    with MetricsCollector(
        client=client,
        processor_instance=processor,
        track_workers=True,
        worker_tracking_interval=1.0,
    ) as collector:
        t0 = time.perf_counter()
        output, report = run_processor_workflow(
            config=validated_config,
            output_manager=output_manager,
            metadata_lookup=metadata_lookup,
            processor=processor,
            workitems=workitems,
            executor=DaskExecutor(client=client, treereduction=8, retries=0),
            schema=NanoAODSchema,
        )
        t1 = time.perf_counter()
        collector.extract_metrics_from_output(output)
        collector.set_coffea_report(report)

    return {
        "label": label,
        "wall_time": t1 - t0,
        "output": output,
        "report": report,
        "metrics": collector.get_metrics(),
        "tracking_data": collector.tracking_data,
        "span_metrics": getattr(collector, "span_metrics", None),
    }


results = {}
with acquire_client(
    AF, close_after=AUTO_CLOSE_CLIENT, pip_packages=WORKER_DEPENDENCIES
) as (client, cluster):
    print("▶ Running HaaS arm (HistServProcessor)…")
    haas_proc = HistServProcessor(
        config=validated_config,
        output_manager=output_manager,
        metadata_lookup=metadata_lookup,
    )
    results["haas"] = run_arm(client, haas_proc, "HaaS (histserv)")

    print("\n▶ Running local arm (HistLocalProcessor)…")
    local_proc = HistLocalProcessor(
        config=validated_config,
        output_manager=output_manager,
        metadata_lookup=metadata_lookup,
    )
    results["local"] = run_arm(client, local_proc, "local reduce")

for key, r in results.items():
    print(f"{r['label']}: {r['wall_time']:.1f}s, {r['output'].get('processed_events', 0):,} events")

## Metrics comparison

In [ ]:
from rich.console import Console
from roastcoffea.export.reporter import (
    format_throughput_table,
    format_event_processing_table,
    format_resources_table,
    format_timing_table,
)

console = Console()

for key in ("haas", "local"):
    r = results[key]
    console.rule(f"[bold]{r['label']}")
    print("📈 Throughput")
    console.print(format_throughput_table(r["metrics"]))
    print("⚡ Event processing")
    console.print(format_event_processing_table(r["metrics"]))
    print("🖥️  Resources")
    console.print(format_resources_table(r["metrics"]))
    print("⏱️  Timing")
    console.print(format_timing_table(r["metrics"]))

### Side-by-side summary
A condensed view pulling the headline numbers from both arms.

In [ ]:
from rich.table import Table

haas = results["haas"]
local = results["local"]

def fmt(v, suffix=""):
    if v is None:
        return "—"
    if isinstance(v, float):
        return f"{v:,.2f}{suffix}"
    return f"{v:,}{suffix}"

rows = [
    ("Wall time (s)", haas["wall_time"], local["wall_time"]),
    ("Events processed",
     haas["output"].get("processed_events", 0),
     local["output"].get("processed_events", 0)),
    ("Throughput (events/s)",
     haas["output"].get("processed_events", 0) / haas["wall_time"],
     local["output"].get("processed_events", 0) / local["wall_time"]),
    ("Bytes read (MB)",
     haas["report"].get("bytesread", 0) / 1e6,
     local["report"].get("bytesread", 0) / 1e6),
    ("Chunks",
     haas["report"].get("chunks"),
     local["report"].get("chunks")),
]

table = Table(title="HaaS vs local summary")
table.add_column("Metric")
table.add_column("HaaS", justify="right")
table.add_column("Local", justify="right")
table.add_column("Δ (local − haas)", justify="right")
for name, h, l in rows:
    delta = (l - h) if (isinstance(h, (int, float)) and isinstance(l, (int, float))) else None
    table.add_row(name, fmt(h), fmt(l), fmt(delta))
console.print(table)

## Histogram correctness check
Pull the HaaS histogram back from the server via `.to_hist()` and compare bin contents to the locally-reduced histogram. Any divergence beyond float noise indicates a bug in one of the paths.

In [ ]:
import numpy as np

def materialize(h):
    """Return a hist.Hist regardless of ChunkedHist vs Hist."""
    return h.to_hist() if hasattr(h, "to_hist") else h

haas_hists = results["haas"]["output"]["histograms"]
local_hists = results["local"]["output"]["histograms"]

mismatches = 0
for channel, by_obs in haas_hists.items():
    for obs, h_haas in by_obs.items():
        h_l = local_hists.get(channel, {}).get(obs)
        if h_l is None:
            print(f"❌ {channel}/{obs}: missing in local output")
            mismatches += 1
            continue
        v_haas = materialize(h_haas).view(flow=True)["value"]
        v_l = materialize(h_l).view(flow=True)["value"]
        if v_haas.shape != v_l.shape:
            print(f"❌ {channel}/{obs}: shape mismatch {v_haas.shape} vs {v_l.shape}")
            mismatches += 1
            continue
        close = np.allclose(v_haas, v_l, rtol=1e-6, atol=1e-9)
        status = "✅" if close else "❌"
        max_abs = float(np.max(np.abs(v_haas - v_l))) if v_haas.size else 0.0
        print(f"{status} {channel}/{obs}: max|Δ|={max_abs:.3e}, sum_haas={v_haas.sum():.2f}, sum_local={v_l.sum():.2f}")
        if not close:
            mismatches += 1

print(f"\n{mismatches} mismatch(es)")